# UJIAN TENGAH SEMESTER (UTS) TEMU KEMBALI INFORMASI
## Tugas 4: Preprocessing, Inverted Indexing, Pembobotan (TF-IDF & BM25), dan Sistem Temu Kembali Informasi

---
**Identitas Mahasiswa:**
- **Nama Mahasiswa** : Adnan Amhar
- **NIM** : 202310370311001 (3 Digit Terakhir: 001)
- **Mata Kuliah** : Temu Kembali Informasi (Information Retrieval)
- **Dosen Pengampu** : Galih W. W
- **Institusi** : Fakultas Teknik, Universitas Muhammadiyah Malang

**Capaian Pembelajaran (SUBCPMK-4):**
- Mampu memahami dan menerapkan model temu kembali informasi serta teknik pembobotan dalam pengindeksan dan pencarian teks.
- Menggunakan **seluruh bagian putusan (full-text)** perkara Pidana Narkotika Pengadilan Negeri Jakarta Timur sebagai dataset korpus mandiri.
---

### 1. Instalasi Pustaka Pemrosesan Teks Bahasa Indonesia
Menginstal pustaka `PySastrawi` untuk stemming Bahasa Indonesia, `pandas` untuk manipulasi data, dan `matplotlib` untuk visualisasi distribusi term.

In [ ]:
!pip install -q PySastrawi pandas matplotlib

### 2. Import Modul & Penyiapan Korpus Dokumen Putusan
Membaca seluruh naskah teks putusan (.txt) dari folder korpus data yang telah diekstrak.

In [ ]:
import os
import re
import math
from collections import defaultdict, Counter
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Direktori korpus teks putusan
calon_path = [Path("data_putusan_55/txt"), Path("data_putusan_50/txt"), Path("../Tugas_3_Crawler_Ekstraksi/data_putusan_55/txt"), Path("../Tugas_3_Crawler_Ekstraksi/data_putusan_50/txt")]
DIR_KORPUS = next((p for p in calon_path if p.exists()), Path("data_putusan_55/txt"))
txt_files = sorted(list(DIR_KORPUS.glob("*.txt")))

korpus_dokumen = {}
for f_path in txt_files:
    with open(f_path, 'r', encoding='utf-8', errors='ignore') as f:
        korpus_dokumen[f_path.stem] = f.read()

print(f"Total dokumen putusan yang dimuat ke korpus: {len(korpus_dokumen)} dokumen.")
if korpus_dokumen:
    contoh_id = list(korpus_dokumen.keys())[0]
    print(f"Contoh dokumen [{contoh_id}] memiliki panjang: {len(korpus_dokumen[contoh_id])} karakter.")

### 3. Pipeline Text Preprocessing (Bahasa Indonesia)
Menerapkan 4 tahapan preprocessing sesuai kaidah Temu Kembali Informasi:
1. **Cleaning & Case Folding**: Menghilangkan watermark MA, disclaimer, nomor halaman, dan karakter non-alfabet.
2. **Tokenizing**: Pemecahan kalimat menjadi daftar kata (token).
3. **Stopword Removal**: Menghilangkan kata umum Bahasa Indonesia (Sastrawi) ditambah *custom legal stopwords* (kata administratif peradilan yang tidak bermakna pembeda).
4. **Stemming Teroptimasi**: Menggunakan kamus memoization (*hash-cache*) agar proses stemming korpus berjalan cepat dalam hitungan detik.

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Setup Sastrawi
stemmer = StemmerFactory().create_stemmer()
stopwords_umum = set(StopWordRemoverFactory().get_stop_words())

# Stopwords administratif peradilan (tidak membuang kata zat narkotika & vonis)
CUSTOM_LEGAL_STOPWORDS = {
    'halaman', 'dari', 'putusan', 'nomor', 'mahkamah', 'agung', 'republik', 'indonesia',
    'direktori', 'pengadilan', 'negeri', 'panitera', 'pengganti', 'hakim', 'ketua', 'anggota',
    'jaksa', 'penuntut', 'umum', 'tersebut', 'atas', 'bahwa', 'oleh', 'karena', 'itu', 'maka',
    'kepada', 'dalam', 'pada', 'untuk', 'dengan', 'dan', 'atau', 'sebagaimana', 'dimaksud',
    'berdasarkan', 'surat', 'tertanggal', 'tahun', 'bulan', 'hari', 'tanggal', 'persidangan',
    'sidang', 'musyawarah', 'dihadiri', 'dibacakan', 'terbuka', 'disampaikan', 'berita', 'acara',
    'register', 'salinan', 'catatan', 'perkara', 'memeriksa', 'mengadili', 'menimbang',
    'mengingat', 'memperhatikan', 'pasal', 'undang', 'ayat', 'kitab', 'hukum', 'acara', 'pidana',
    'kuhap', 'biaya', 'sejumlah', 'rupiah', 'demikian', 'demikianlah', 'ditetapkan', 'diucapkan'
}

SEMUA_STOPWORDS = stopwords_umum.union(CUSTOM_LEGAL_STOPWORDS)

# Caching kamus kata dasar agar proses stemming ribuan kata berjalan instan
STEM_CACHE = {}

def fast_stem(kata):
    if kata not in STEM_CACHE:
        STEM_CACHE[kata] = stemmer.stem(kata)
    return STEM_CACHE[kata]

def preprocessing_teks(teks, use_stemming=True):
    # 1. Case folding & Cleaning
    teks_lower = teks.lower()
    teks_clean = re.sub(r'disclaimer\s+kepaniteraan[^
]*', ' ', teks_lower)
    teks_clean = re.sub(r'halaman\s+\d+\s+dari\s+\d+[^
]*', ' ', teks_clean)
    teks_clean = re.sub(r'[^a-z\s]', ' ', teks_clean)
    
    # 2. Tokenizing
    tokens = [t.strip() for t in teks_clean.split() if len(t.strip()) > 1]
    
    # 3. Stopword Removal
    tokens_no_stop = [t for t in tokens if t not in SEMUA_STOPWORDS]
    
    # 4. Stemming dengan Cache
    if use_stemming:
        tokens_final = [fast_stem(t) for t in tokens_no_stop if len(fast_stem(t)) > 1]
    else:
        tokens_final = tokens_no_stop
        
    return tokens_final

print("Pipeline preprocessing siap digunakan.")

### 4. Menjalankan Preprocessing pada Seluruh Korpus
Mengekstrak token kata bersih dari setiap dokumen putusan narkotika.

In [ ]:
korpus_tokens = {}
doc_lengths = {}

print("Memulai preprocessing korpus dokumen putusan...")
for doc_id, raw_text in korpus_dokumen.items():
    tokens = preprocessing_teks(raw_text, use_stemming=True)
    korpus_tokens[doc_id] = tokens
    doc_lengths[doc_id] = len(tokens)

print(f"Preprocessing selesai! Jumlah kata unik di cache: {len(STEM_CACHE)} kata.")

### 5. Pembangunan Inverted Index & Pembobotan Term
Membangun struktur data **Inverted Index** yang memetakan setiap term unik ke daftar dokumen beserta frekuensi kemunculannya (`term` -> `{doc_id: tf}`). Menghitung nilai **TF-IDF** dan **Okapi BM25**.

In [ ]:
inverted_index = defaultdict(dict)
vocab = set()

for doc_id, tokens in korpus_tokens.items():
    tf_counter = Counter(tokens)
    for term, freq in tf_counter.items():
        inverted_index[term][doc_id] = freq
        vocab.add(term)

N_DOCS = len(korpus_tokens)
avg_doc_len = sum(doc_lengths.values()) / N_DOCS if N_DOCS > 0 else 0

# Hitung IDF (Inverse Document Frequency)
idf_dict = {}
for term, postings in inverted_index.items():
    df_t = len(postings)
    idf_dict[term] = math.log((N_DOCS - df_t + 0.5) / (df_t + 0.5) + 1.0)

print(f"=== STATISTIK INDEX TEMU KEMBALI INFORMASI ===")
print(f"Total Dokumen (N)            : {N_DOCS}")
print(f"Ukuran Kosakata (Vocabulary) : {len(vocab)} term")
print(f"Rata-rata Panjang Dokumen    : {avg_doc_len:.1f} token")

### 6. Visualisasi 15 Term Paling Sering Muncul dalam Korpus
Menampilkan grafik distribusi term kata kunci terbanyak dalam kasus peradilan narkotika PN Jakarta Timur.

In [ ]:
total_tf = Counter()
for postings in inverted_index.values():
    for doc_id, count in postings.items():
        pass

term_freqs = {term: sum(postings.values()) for term, postings in inverted_index.items()}
top_15_terms = sorted(term_freqs.items(), key=lambda x: x[1], reverse=True)[:15]

plt.figure(figsize=(10, 5))
terms, freqs = zip(*top_15_terms)
plt.barh(terms[::-1], freqs[::-1], color='#1F4E78')
plt.title('Top 15 Term Paling Dominan pada Putusan Narkotika PN Jkt.Tim')
plt.xlabel('Total Kemunculan (Frekuensi)')
plt.tight_layout()
plt.show()

### 7. Engine Temu Kembali Informasi (Search & Ranking Engine)
Mengimplementasikan fungsi pencarian dokumen menggunakan dua algoritma pemeringkatan utama:
1. **Okapi BM25 Ranking**
2. **Vector Space Model (VSM) Cosine Similarity Ranking**

In [ ]:
def cari_dokumen_bm25(query, k1=1.5, b=0.75, top_k=5):
    """Mencari dan memeringkat dokumen berdasarkan skor BM25."""
    q_tokens = preprocessing_teks(query, use_stemming=True)
    skor_docs = defaultdict(float)
    
    for term in q_tokens:
        if term not in inverted_index:
            continue
        idf = idf_dict.get(term, 0.0)
        for doc_id, tf in inverted_index[term].items():
            d_len = doc_lengths[doc_id]
            num = tf * (k1 + 1)
            den = tf + k1 * (1 - b + b * (d_len / avg_doc_len))
            skor_docs[doc_id] += idf * (num / den)
            
    ranked = sorted(skor_docs.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    hasil = []
    for doc_id, score in ranked:
        # Cari cuplikan teks pendukung
        raw_doc = korpus_dokumen[doc_id]
        cuplikan = re.sub(r'\s+', ' ', raw_doc[:220]) + '...'
        hasil.append({
            'Peringkat': len(hasil) + 1,
            'No Putusan (Doc ID)': doc_id,
            'Skor BM25': round(score, 4),
            'Cuplikan Teks': cuplikan
        })
    return pd.DataFrame(hasil)

def cari_dokumen_cosine(query, top_k=5):
    """Mencari dan memeringkat dokumen berdasarkan Cosine Similarity."""
    q_tokens = preprocessing_teks(query, use_stemming=True)
    if not q_tokens:
        return pd.DataFrame()
        
    q_tf = Counter(q_tokens)
    q_vec = {t: (1 + math.log(f)) * idf_dict.get(t, 0.0) for t, f in q_tf.items()}
    q_norm = math.sqrt(sum(v**2 for v in q_vec.values()))
    if q_norm == 0:
        return pd.DataFrame()
        
    dot_prods = defaultdict(float)
    for term, q_weight in q_vec.items():
        if term not in inverted_index:
            continue
        for doc_id, tf in inverted_index[term].items():
            d_weight = (1 + math.log(tf)) * idf_dict.get(term, 0.0)
            dot_prods[doc_id] += q_weight * d_weight
            
    ranked = []
    for doc_id, dot in dot_prods.items():
        d_norm = math.sqrt(sum(((1 + math.log(tf)) * idf_dict.get(t, 0.0))**2 
                              for t, tf in Counter(korpus_tokens[doc_id]).items()))
        if d_norm > 0:
            ranked.append((doc_id, dot / (q_norm * d_norm)))
            
    ranked = sorted(ranked, key=lambda x: x[1], reverse=True)[:top_k]
    
    hasil = []
    for doc_id, score in ranked:
        raw_doc = korpus_dokumen[doc_id]
        cuplikan = re.sub(r'\s+', ' ', raw_doc[:220]) + '...'
        hasil.append({
            'Peringkat': len(hasil) + 1,
            'No Putusan (Doc ID)': doc_id,
            'Cosine Similarity': round(score, 4),
            'Cuplikan Teks': cuplikan
        })
    return pd.DataFrame(hasil)

### 8. Pengujian Query Pencarian Perkara Hukum Narkotika
Menguji efektivitas sistem penelusuran temu kembali informasi menggunakan 3 skenario query hukum berbeda.

In [ ]:
# Skenario 1: Pencarian berdasarkan jenis barang bukti dan kemasan
query_1 = "sabu dalam plastik klip dan timbangan digital"
print(f"=== HASIL PENCARIAN BM25 (Query: '{query_1}') ===")
display(cari_dokumen_bm25(query_1, top_k=3))

# Skenario 2: Pencarian berdasarkan amar vonis penjara dan denda
query_2 = "menjatuhkan pidana penjara denda satu milyar rupiah subsider"
print(f"\n=== HASIL PENCARIAN COSINE SIMILARITY (Query: '{query_2}') ===")
display(cari_dokumen_cosine(query_2, top_k=3))

# Skenario 3: Pencarian pasal tindak pidana narkotika
query_3 = "tindak pidana narkotika pasal 114 ayat 1 pemufakatan jahat"
print(f"\n=== HASIL PENCARIAN BM25 (Query: '{query_3}') ===")
display(cari_dokumen_bm25(query_3, top_k=3))